<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/Nam-Wan/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_book_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import random
from datetime import datetime

# 1. โหลดข้อมูลจาก CSV
url = 'https://raw.githubusercontent.com/dranphphmithe-ux/Book-Rental-System-Project/refs/heads/main/books_cleaned.csv'
df_books = pd.read_csv(url)
df_books.columns = df_books.columns.str.strip()

# 2. รายชื่อลูกค้าสำหรับออกใบเสร็จ
FIRST_NAMES = ["กิตติพงษ์", "ณิชา", "ธนกฤต", "ปรียา", "พงศกร", "ภัทรวดี", "วรวุฒิ", "ศิริพร", "อัครพล", "อนันดา"]
LAST_NAMES = ["ใจดี", "เจริญสุข", "สมบูรณ์", "วงษ์สุวรรณ", "รัตนไพศาล", "พงษ์พาณิชย์", "ชินวัตร", "ทองแท้", "สุวรรณรัตน์", "มั่นคง"]

def print_receipt(customer_name, customer_id, initial_points, items, days_rented, days_late):
    order_id = f"REC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000, 9999)}"
    date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    total_books = len(items)

    # 1. คำนวณค่ายืมปกติต่อเล่ม (3 วัน 20 บาท, เศษวันละ 7 บาท)
    sets_of_3 = days_rented // 3
    remaining_days = days_rented % 3
    rental_fee_per_book = (sets_of_3 * 20) + (remaining_days * 7)
    total_rental_before_discount = total_books * rental_fee_per_book

    # 2. แต้มและส่วนลด (1 เล่ม = 1 แต้ม, คืนตรงเวลา +1 แต้ม, คืนช้าไม่ได้แต้ม)
    base_points = total_books
    on_time_bonus = 1 if days_late == 0 else 0
    earned_points = base_points + on_time_bonus
    total_points_accumulated = initial_points + earned_points

    # ส่วนลด: ครบ 10 แต้ม ได้อ่านฟรี 1 วัน (ลด 7 บาท/เล่ม)
    free_books_count = min(total_points_accumulated // 10, total_books)
    total_discount = free_books_count * 7.0

    # 3. ค่าปรับและสรุปยอด
    total_rental_fee = max(0.0, total_rental_before_discount - total_discount)
    total_fine = total_books * days_late * 10
    grand_total = total_rental_fee + total_fine

    # อัปเดตแต้มคงเหลือ
    points_used = free_books_count * 10
    final_points = total_points_accumulated - points_used

    # --- แสดงผลใบเสร็จ ---
    print("=" * 55)
    print(f"{'ใบเสร็จรับเงิน / Receipt':^55}")
    print("=" * 55)
    print(f"เลขที่ใบเสร็จ: {order_id}")
    print(f"วันที่ออกใบเสร็จ: {date_issued}")
    print(f"ชื่อลูกค้า: {customer_name} (ID: {customer_id})")
    print(f"จำนวนวันที่ยืม: {days_rented} วัน")
    print(f"สถานะการคืน: {'⚠️ คืนช้า ' + str(days_late) + ' วัน' if days_late > 0 else '✅ คืนตรงเวลา'}")
    print("-" * 55)

    print(f"รายการหนังสือที่ยืม ({total_books} เล่ม):")
    for i, item in enumerate(items, 1):
        print(f"  [{i:02d}] {item['title']} (ราคาปก {item['price']:.0f} บาท)")

    print("-" * 55)
    print(f"อัตราค่ายืมปกติต่อเล่ม ({days_rented} วัน): {rental_fee_per_book:.2f} บาท")

    if free_books_count > 0:
        print(f"🎁 ส่วนลดสะสมแต้ม (ครบ 10 แต้ม): อ่านฟรี 1 วัน จำนวน {free_books_count} เล่ม (-{total_discount:.2f} บาท)")

    print(f"รวมค่ายืมหนังสือหลังหักส่วนลด: {total_rental_fee:.2f} บาท")

    if total_fine > 0:
        print(f"❌ ค่าปรับคืนช้า ({days_late} วัน x {total_books} เล่ม x 10B): {total_fine:.2f} บาท")

    print("-" * 55)
    print(f"ยอดชำระสุทธิ (Grand Total): {grand_total:.2f} บาท")
    print("-" * 55)

    print(f"✨ แต้มที่ได้รับจากจำนวนหนังสือ (1 เล่ม = 1 แต้ม): +{base_points} แต้ม")
    if days_late == 0:
        print(f"🎉 โบนัสคืนตรงเวลา: +{on_time_bonus} แต้ม")
    else:
        print("🚫 คืนช้ากว่ากำหนด: ไม่ได้รับโบนัสคืนตรงเวลา (+0 แต้ม)")

    if points_used > 0:
        print(f"🔄 ใช้แต้มแลกอ่านฟรี: -{points_used} แต้ม")

    print(f"🏆 แต้มสะสมคงเหลือปัจจุบัน: {final_points} แต้ม")
    print("=" * 55 + "\n")

# ==========================================
# ตัวอย่างการพิมพ์ใบเสร็จ 1 รายการ
# ==========================================
# 1. เตรียมข้อมูลหนังสือที่เลือก
sampled = df_books.sample(n=random.randint(1, 5))
items_list = []
for _, row in sampled.iterrows():
    title = str(row.get('title', row.get('book_title', row.get('name', 'หนังสือทั่วไป'))))
    price = float(row.get('price', 100))
    items_list.append({'title': title, 'price': price})

# 2. พิมพ์ใบเสร็จ
print_receipt(
    customer_name=f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}",
    customer_id=f"C{random.randint(100, 999)}",
    initial_points=random.choice([0, 5, 8, 10, 12]),
    items=items_list,
    days_rented=3, # จำนวนวันที่ยืม
    days_late=0    # จำนวนวันที่คืนช้า
)

               ใบเสร็จรับเงิน / Receipt                
เลขที่ใบเสร็จ: REC-20260830-5743
วันที่ออกใบเสร็จ: 2026-08-30 08:35:34
ชื่อลูกค้า: ภัทรวดี เจริญสุข (ID: C771)
จำนวนวันที่ยืม: 3 วัน
สถานะการคืน: ✅ คืนตรงเวลา
-------------------------------------------------------
รายการหนังสือที่ยืม (4 เล่ม):
  [01] ล่าขุมทรัพย์สุดขอบฟ้า เล่ม 40 (ราคาปก 165 บาท)
  [02] Spy x Family เล่ม 11 (ราคาปก 95 บาท)
  [03] เรื่องผีรอบโลก เล่ม 14 (ราคาปก 165 บาท)
  [04] ครอบครัวตึ๋งหนืด เล่ม 19 (ราคาปก 165 บาท)
-------------------------------------------------------
อัตราค่ายืมปกติต่อเล่ม (3 วัน): 20.00 บาท
🎁 ส่วนลดสะสมแต้ม (ครบ 10 แต้ม): อ่านฟรี 1 วัน จำนวน 1 เล่ม (-7.00 บาท)
รวมค่ายืมหนังสือหลังหักส่วนลด: 73.00 บาท
-------------------------------------------------------
ยอดชำระสุทธิ (Grand Total): 73.00 บาท
-------------------------------------------------------
✨ แต้มที่ได้รับจากจำนวนหนังสือ (1 เล่ม = 1 แต้ม): +4 แต้ม
🎉 โบนัสคืนตรงเวลา: +1 แต้ม
🔄 ใช้แต้มแลกอ่านฟรี: -10 แต้ม
🏆 แต้มสะสมคงเหลือปัจจุบัน: 3 แ